# ShadeGap DFW 2.0 — same-year (summer 2026) stop attributes
GISC 4363/6363 Final Project · Samia Riaz

Addresses the midterm comment: **bus stops, tree canopy and surface temperature all come from 2026.**

| Layer | Source | Period |
|---|---|---|
| Bus stops | DART GTFS (current 2026 feed) | 2026 |
| Tree canopy | Dynamic World V1 `trees` (10 m) | Jun–Aug 2026 |
| Surface temperature | Landsat 8/9 C2 L2 `ST_B10` | Jun–Aug 2026 |
| Social vulnerability | CDC/ATSDR SVI 2022 (census-based, newest release) | 2022 |
| Validation | USFS Tree Canopy Cover 2023 vs Dynamic World 2023 | 2023 |

Run the cells top to bottom. Outputs go to `/content/shadegap_out/` and the two rasters go to Google Drive (`ShadeGap_Final/`).

In [ ]:
!pip -q install earthengine-api geopandas
import ee, io, zipfile, os, requests, json, time
import pandas as pd, numpy as np, geopandas as gpd
import matplotlib.pyplot as plt

ee.Authenticate()
ee.Initialize(project='ee-ranamobeen269')
OUT = '/content/shadegap_out'; os.makedirs(OUT, exist_ok=True)
print('Earth Engine ready')

## Settings

In [ ]:
START, END = '2026-06-01', '2026-09-01'   # end is exclusive -> June, July, August
STOP_BUFFER_M  = 30      # waiting-area radius around each stop (m)
TREE_THRESHOLD = 0.5     # Dynamic World trees probability counted as canopy
# Heat Exposure Index weights (must sum to 1) — same formula reused in the web app
W_HEAT, W_SHADE, W_SVI = 0.4, 0.3, 0.3
DRIVE_FOLDER = 'ShadeGap_Final'
GTFS_URL = 'https://www.dart.org/transitdata/latest/google_transit.zip'
SVI_URL  = 'https://svi.cdc.gov/Documents/Data/2022/csv/states/Texas.csv'

## 1. DART stops from GTFS (with the modes serving each stop)

In [ ]:
r = requests.get(GTFS_URL, timeout=120); r.raise_for_status()
z = zipfile.ZipFile(io.BytesIO(r.content))
feed_info = pd.read_csv(z.open('feed_info.txt'), dtype=str) if 'feed_info.txt' in z.namelist() else None
if feed_info is not None:
    print(feed_info[[c for c in feed_info.columns if 'version' in c or 'date' in c]].to_string(index=False))

stops  = pd.read_csv(z.open('stops.txt'),  dtype={'stop_id': str})
routes = pd.read_csv(z.open('routes.txt'), dtype={'route_id': str})
trips  = pd.read_csv(z.open('trips.txt'),  dtype={'route_id': str, 'trip_id': str}, usecols=['route_id','trip_id'])
st     = pd.read_csv(z.open('stop_times.txt'), dtype={'trip_id': str, 'stop_id': str}, usecols=['trip_id','stop_id'])

# boarding points only (location_type 0 or blank)
if 'location_type' in stops.columns:
    stops = stops[stops['location_type'].fillna(0).astype(int) == 0]

MODE = {0: 'Light rail/Streetcar', 1: 'Subway', 2: 'Rail', 3: 'Bus'}
sr = (st.drop_duplicates()
        .merge(trips, on='trip_id')
        .merge(routes[['route_id','route_type','route_short_name']], on='route_id')
        .drop_duplicates(['stop_id','route_id']))
agg = sr.groupby('stop_id').agg(
    n_routes=('route_id','nunique'),
    routes=('route_short_name', lambda s: ', '.join(sorted(map(str, s.unique())))[:250]),
    is_bus=('route_type', lambda s: int((s == 3).any())),
    modes=('route_type', lambda s: ', '.join(sorted({MODE.get(int(v), str(v)) for v in s}))))
# trips per stop (service level, all service days in the feed)
trips_per_stop = st.groupby('stop_id').size().rename('trips_in_feed')

stops = (stops[['stop_id','stop_name','stop_lat','stop_lon']]
         .merge(agg, on='stop_id', how='inner')          # drops stops with no service
         .merge(trips_per_stop, on='stop_id', how='left'))
print(f'{len(stops):,} served stops  |  bus stops: {stops.is_bus.sum():,}')
stops.head()

## 2. SVI 2022 (Texas tracts) joined to Census tract polygons in Earth Engine

In [ ]:
try:
    svi = pd.read_csv(SVI_URL, dtype={'FIPS': str})
except Exception as e:
    print('Direct download failed (', e, ') — upload Texas.csv from https://www.atsdr.cdc.gov/place-health/php/svi/svi-data-documentation-download.html')
    from google.colab import files
    up = files.upload(); svi = pd.read_csv(io.BytesIO(next(iter(up.values()))), dtype={'FIPS': str})
svi = svi[['FIPS','RPL_THEMES']]
svi = svi[svi.RPL_THEMES >= 0]                       # -999 = no data
svi['FIPS'] = svi.FIPS.str.zfill(11)
print(f'{len(svi):,} Texas tracts with SVI')

## 3. Send stops to Earth Engine and build the 2026 layers

In [ ]:
def to_fc(df):
    feats = [ee.Feature(ee.Geometry.Point([float(r.stop_lon), float(r.stop_lat)]), {'stop_id': r.stop_id})
             for r in df.itertuples()]
    return ee.FeatureCollection(feats)

stops_fc = to_fc(stops)
aoi = ee.Geometry.Rectangle([stops.stop_lon.min()-0.03, stops.stop_lat.min()-0.03,
                             stops.stop_lon.max()+0.03, stops.stop_lat.max()+0.03])

# --- Landsat 8/9 land surface temperature, summer 2026 ---
def mask_lst(img):
    qa = img.select('QA_PIXEL')
    clear = (qa.bitwiseAnd(1 << 1).eq(0)      # dilated cloud
             .And(qa.bitwiseAnd(1 << 3).eq(0))  # cloud
             .And(qa.bitwiseAnd(1 << 4).eq(0))) # cloud shadow
    return (img.select('ST_B10').multiply(0.00341802).add(149.0).subtract(273.15)
              .rename('LST_C').updateMask(clear))
landsat = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
           .merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2'))
           .filterBounds(aoi).filterDate(START, END).filter(ee.Filter.lt('CLOUD_COVER', 40)))
lst = landsat.map(mask_lst).median().clip(aoi)

# --- Dynamic World trees, summer 2026 ---
dw = ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1').filterBounds(aoi).filterDate(START, END)
trees_prob = dw.select('trees').median().clip(aoi).rename('trees_prob')

# --- SVI as raster: TIGER 2020 tracts + SVI dictionary by GEOID ---
svi_dict = ee.Dictionary(dict(zip(svi.FIPS, svi.RPL_THEMES.astype(float))))
tracts = (ee.FeatureCollection('TIGER/2020/TRACT').filterBounds(aoi)
          .filter(ee.Filter.inList('GEOID', svi_dict.keys())))
tracts = tracts.map(lambda f: f.set('RPL', svi_dict.get(f.get('GEOID'))))
svi_img = tracts.reduceToImage(['RPL'], ee.Reducer.first()).rename('SVI_RPL')

# --- Validation: 2023 for both products ---
tcc2023 = (ee.ImageCollection('USGS/NLCD_RELEASES/2023_REL/TCC/v2023-5')
           .filterDate('2023-01-01', '2024-01-01').filterBounds(aoi)
           .select('NLCD_Percent_Tree_Canopy_Cover').mosaic().rename('TCC2023_pct'))
dw2023 = (ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1').filterBounds(aoi)
          .filterDate('2023-06-01', '2023-09-01').select('trees').median()
          .gt(TREE_THRESHOLD).rename('DW2023_bin'))

print('Landsat scenes:', landsat.size().getInfo(), '| Dynamic World images:', dw.size().getInfo())
stack = lst.addBands(trees_prob).addBands(svi_img).addBands(tcc2023).addBands(dw2023)

## 4. Sample every stop (30 m waiting area), in chunks

In [ ]:
rows, CH = [], 500
for i in range(0, len(stops), CH):
    fc = to_fc(stops.iloc[i:i+CH]).map(lambda f: f.buffer(STOP_BUFFER_M))
    res = stack.reduceRegions(collection=fc, reducer=ee.Reducer.mean(), scale=10, tileScale=4).getInfo()
    rows += [f['properties'] for f in res['features']]
    print(f'{min(i+CH, len(stops)):,}/{len(stops):,}', end='  ')
samp = pd.DataFrame(rows)
df = stops.merge(samp, on='stop_id', how='left').rename(columns={
    'LST_C': 'lst_c', 'trees_prob': 'trees_prob', 'SVI_RPL': 'svi_rpl', 'TCC2023_pct': 'tcc2023_pct'})
df = df.drop(columns=['DW2023_bin'], errors='ignore')
df['data_year'] = 2026
print('\nMissing values:\n', df[['lst_c','trees_prob','svi_rpl']].isna().sum())

## 5. Heat Exposure Index (HEI)
Shade is measured by the **tree index** = mean Dynamic World `trees` probability in the 30 m waiting area (×100).
(A 0.5 canopy threshold was tested first, but 99% of stops then read 0% canopy, so the continuous probability is used.)

`HEI = 100 × (W_HEAT·pct(LST) + W_SHADE·(1 − pct(tree index)) + W_SVI·SVI)` — percentile ranks across all bus stops.

In [ ]:
bus = df[df.is_bus == 1].dropna(subset=['lst_c','trees_prob','svi_rpl']).copy()
bus['tree_idx'] = (bus.trees_prob * 100).round(2)
bus['pct_lst']  = bus.lst_c.rank(pct=True).round(4)
bus['pct_tree'] = bus.tree_idx.rank(pct=True).round(4)
bus['hei'] = (100 * (W_HEAT*bus.pct_lst + W_SHADE*(1 - bus.pct_tree) + W_SVI*bus.svi_rpl)).round(1)
bus['hei_rank'] = bus.hei.rank(ascending=False, method='min').astype(int)
bus['priority'] = pd.cut(bus.hei, [-1, 40, 55, 70, 101], labels=['Low','Moderate','High','Very high']).astype(str)
print(bus.priority.value_counts())
bus.sort_values('hei', ascending=False).head(10)[['stop_name','routes','lst_c','tree_idx','svi_rpl','hei','priority']]

## 6. Check the shade measure against USFS Tree Canopy Cover 2023

In [ ]:
from scipy.stats import spearmanr
rho = spearmanr(bus.tcc2023_pct, bus.tree_idx).statistic
fig, ax = plt.subplots(figsize=(6,5))
ax.scatter(bus.tcc2023_pct, bus.tree_idx, s=5, alpha=.3, color='#2a7d4f')
ax.set_xlabel('USFS Tree Canopy Cover 2023 (%)'); ax.set_ylabel('Dynamic World tree index, summer 2026')
ax.set_title(f'Shade measure check at {len(bus):,} DART bus stops (Spearman rho = {rho:.2f})', fontsize=10)
plt.tight_layout(); plt.savefig(f'{OUT}/validation_tree_index.png', dpi=200); plt.show()

## 7. Save outputs (CSV + GeoJSON for ArcGIS Online, app reference file)

In [ ]:
keep = ['stop_id','stop_name','stop_lat','stop_lon','routes','n_routes','modes','trips_in_feed',
        'lst_c','tree_idx','svi_rpl','pct_lst','pct_tree','hei','hei_rank','priority','tcc2023_pct','data_year']
bus[keep].to_csv(f'{OUT}/shadegap_bus_stops_2026.csv', index=False)
gpd.GeoDataFrame(bus[keep], geometry=gpd.points_from_xy(bus.stop_lon, bus.stop_lat), crs=4326)\
   .to_file(f'{OUT}/shadegap_bus_stops_2026.geojson', driver='GeoJSON')
ref = {'weights': {'heat': W_HEAT, 'shade': W_SHADE, 'svi': W_SVI},
       'shade_field': 'tree_idx (Dynamic World trees probability x100, 30 m mean)',
       'lst_sorted': sorted(bus.lst_c.round(2).tolist()),
       'tree_sorted': sorted(bus.tree_idx.round(2).tolist()),
       'period': '2026-06-01 to 2026-08-31', 'buffer_m': STOP_BUFFER_M}
json.dump(ref, open(f'{OUT}/hei_reference.json', 'w'))
print(os.listdir(OUT))

## 8. Rasters for the app's Swipe (Cloud-Optimized GeoTIFF → Google Drive)

In [ ]:
tasks = [
  ee.batch.Export.image.toDrive(image=lst.multiply(10).round().toInt16().unmask(-32768),
      description='ShadeGap_LST_2026_cog', folder=DRIVE_FOLDER, region=aoi, scale=30, crs='EPSG:3857',
      maxPixels=1e10, formatOptions={'cloudOptimized': True, 'noData': -32768}),
  ee.batch.Export.image.toDrive(image=trees_prob.multiply(100).round().toUint8().unmask(255),
      description='ShadeGap_Trees_2026_cog', folder=DRIVE_FOLDER, region=aoi, scale=10, crs='EPSG:3857',
      maxPixels=1e10, formatOptions={'cloudOptimized': True, 'noData': 255})]
for t in tasks: t.start()
while any(t.status()['state'] in ('READY','RUNNING') for t in tasks):
    print([t.status()['state'] for t in tasks]); time.sleep(60)
print([(t.status()['description'], t.status()['state']) for t in tasks])

## 9. Download the results

In [ ]:
!cd /content && zip -qr shadegap_out.zip shadegap_out
from google.colab import files
files.download('/content/shadegap_out.zip')